In [1]:
# Import directories
import numpy as np  # Essential functions
import matplotlib.pyplot as plt # Essential data visualization
import pandas as pd # Essential for dataframe and excel manipulation
from scipy.signal import savgol_filter # For smoothing NADH concentration

# os for handling working folder
import os as os
import os.path as ospath # os.path used to check if a csv exists.

from skimage.io import imread, imsave #Image processing

from PIL import Image # For finding pixel length
from PIL.TiffTags import TAGS # For finding pixel length

In [2]:
# Change directory to master directory to make ease of use possible.
os.chdir('C://Users//franc//Box//ALAB Data Francis Cavanna//2021 Actomyosin Control//')

# Create list of folders I want to process
base = "C://Users//franc//Box//ALAB Data Francis Cavanna//2021 Actomyosin Control" # Base file location

List_folder = ['2022_1_28', '2022_1_31', '2022_2_25','2022_3_13_Anthony//3-13-2022//0,6_F-0,12_M-1', 
               '2022_3_13_Anthony//3-13-2022//0,6_F-0,12_M-2','2022_3_13_Anthony//3-13-2022//0,6_F-0,12_M-3',
              '2022_3_13_Anthony//3-13-2022//1,2 F-0,06 M-1', '2022_3_13_Anthony//3-13-2022//1,2 F-0,12 M-1',
               '2022_3_15_Anthony//1,2 F-0,24 M-1','2022_3_15_Anthony//1,2 F-0,24 M-2','2022_4_16','2023_2_03//fascin']
                # List of folders containing data you would want to process

#DAPI_folder = '\StrainRateFiles\DAPI\\' # Select relevant PLUM folders

#PLUM_folder = '\StrainRateFiles\PLUM\\' # Select relevant DAPI folders

List_folder_Reduced = ['2022_1_28', '2022_1_31', '2022_2_25',
              '2022_3_13_Anthony//3-13-2022//1,2 F-0,06 M-1', '2022_3_13_Anthony//3-13-2022//1,2 F-0,12 M-1',
               '2022_3_15_Anthony//1,2 F-0,24 M-1','2022_3_15_Anthony//1,2 F-0,24 M-2','2022_4_16']
                # List of folders containing data you would want to process, without the extra fascin
List_folder_TTest = ['2022_1_28','2022_3_15_Anthony//1,2 F-0,24 M-1','2022_3_15_Anthony//1,2 F-0,24 M-2',
                    "2022_10_20//Actin_12uM_Fascin_10_Myosin_50_1","2022_10_20//Actin_12uM_Fascin_10_Myosin_50_2",
              "2022_10_22"]

List_Sheets = ['RF=1_10  RM=1_50','RF=1_10  RM=1_200',
                'RF=1_10  RM=1_100','RF=1_20  RM=1_100',
                'RF=1_20  RM=1_100','RF=1_20  RM=1_100',
                'RF=1_10  RM=1_200','RF=1_10  RM=1_100',
                'RF=1_10  RM=1_50','RF=1_10  RM=1_50',
               'RF=1_10  RM=1_100','RF=1_10  RM=1_200'] 


In [3]:
# Define necessary filenames

# This block assigns the directory we work in, and the filenames we generate and use.

# Specify the overarching path for our filenames.
path = "C:\\Users\\franc\\Box\\ALAB Data Francis Cavanna\\2021 Actomyosin Control\\"

# Specify originator Excel Document from MolarConcentration.nb program:
MCfilename = "MathematicaDF.csv"

suffix = '.tif'

#Tiffilename = "1,2F-0,24M-Blue-2 Reslice1"

ECfolder = 'StrainRateFiles\\EnergyConsumpVideos\\'

ED_folder = 'StrainRateFiles\\Energy_Difference\\'

subfolder_base = "\StrainRateFiles\DAPI_Integration\\"

# Set fontsize 
ftsz = 20
tcksz = 14

In [4]:
##############              DEFINE FUNCTIONS                  ##########################

In [5]:
## Import tif datasets:

# Input is to specify dataset number to process
def compileFolders(i):
    filenames = []
    filename = List_folder[i] + subfolder_base
    DapiNames = os.listdir(filename)
    
    # Collect kymograph names from "DAPI_Integration" folder.
    for n in np.arange(0,3):
        
        filename = List_folder[i] + subfolder_base + DapiNames[n] #+ number1d + suffix
        filenames.append(filename)
        
    # Load files as manipulable images
    C1images = [] 
    for fn in filenames:
            im = imread(fn)
            C1images.append(im)
    
    # Return filenames, images
    return filenames,C1images


In [6]:
# Find shortest kymograph length in lists
def shortL(files):
    pixL = []
    for fn in files:
        img = Image.open(fn)
        meta_dict = {TAGS[key] : img.tag[key] for key in img.tag_v2}
        #pixL.append(int(np.ceil(meta_dict['ImageWidth'][0]/10)))
        pixL.append(int(np.ceil(meta_dict['ImageWidth'][0])))
    return min(pixL)



In [7]:
# Iterate through array of filenames and create one averaged master array of microscope intensity values
def avgCond(images,minP): #Take an input of several kymographs, and the minimum length between them.
    MasInt = []
    for j in range(len(images[0])):
    #for j in range(len(images[0][::10])):
        StepInt = []
        for i in range(minP):
            img_List = []
            for k in range(len(images)):
                img_List.append(images[k][j][i])
                #img_List.append(images[k][::9][j][i])
            StepInt.append(np.average(img_List))
        MasInt.append(StepInt)
    return MasInt

In [8]:
# Fits from Mathematica MolarConcentration_Full_NADH_Calibration.nb document.
Fit1 = -0.12123625148063022
Fit2 = 0.00046298638444968403 # Gives NADH concentration in mM

#Create a function that returns NADH concentration from Microscope Intensity
def to_NADH(M_List):
    MsrLin = []
    for j in range(len(M_List)):
        StepInt = []
        for i in range(len(M_List[0])):
            StepInt.append(Fit1 + Fit2*M_List[j][i])
        # Apply a Savitzky-Golay filter to smooth our data
        StepInt = savgol_filter(StepInt, 51, 3)
        MsrLin.append(StepInt)
    return MsrLin


In [9]:
# Make an array of distances in microns
def micronArr(minP):
    # Make array of distance in microns
    microns_Val = []
    for i in range(0,minP):
        dist = to_um(i*1.0)
        microns_Val.append(dist)
    return microns_Val
        
# Get micron size measurements
# X Resolution:  0.7692 pixels per micron
convert_p = 1/0.7692 #conversion factor for microns per pixel.

def to_um(P):
    return convert_p*P

In [10]:
# Create a figure in the appropriate place to measure the experiment results by eye.
def saveFig(SampleCurves,microns_Val,i):
    
    fig = plt.figure(figsize=(8,6))
    ax1 = fig.add_subplot(111)

    # Plot individual curves at different times.
    for k in range(len(SampleCurves)):
         ax1.plot(microns_Val, SampleCurves[k],label="t = "+str(k*200)+"$\it{s}$") # 

    # Set label for y-axis and adjust position
    plt.ylabel('[NADH] (mM)',fontsize=ftsz,rotation=0)
    ax1.yaxis.set_label_coords(-0.1,1.02)

    # Set label for x-axis and adjust position
    plt.xlabel('$distance \: (\mathrm{\mu m})$',fontsize=ftsz)

    #Create legend for time parameter
    plt.legend(bbox_to_anchor=(1.32, 1),loc="upper right",prop={'size': 15})

    # Create title to save conditions of experiment:
    plt.title(List_Sheets[i])
    
    # Enlarge tick parameters
    plt.tick_params(axis='both', which='major', labelsize=tcksz)
    
    # Shade out grey region for supplementary information. Comment out as needed.
    plt.axvspan(microns_Val[-50], microns_Val[-1], color='gray', alpha=0.2)

    # Save the figure
    plt.savefig('C://Users//franc//Box//ALAB Data Francis Cavanna//2021 Actomyosin Control//' 
                + "ProcessingStrainRate_Files\\Results\\NADH_Micron\\" +List_Sheets[i]+'_NADH_vs_Distance_Greyed'+
                str(i)+'.png',dpi="figure",format="png",bbox_inches='tight')

In [11]:
# Save data as excel spreadsheets
def saveData(SampleCurves,microns_Val,i):
    df1 = pd.DataFrame(SampleCurves,index=["t="+str(x*20)+"s" for x in list(pd.RangeIndex(0, 91))])#Saves the NADH concentration
    df2 = pd.DataFrame(microns_Val,columns=["L (microns)"])#Saves the radial distance in data
    df1 = df1.transpose()
    df3 = pd.concat([df1,df2],axis=1)
    #print(df3)
    if ospath.exists(base + "ProcessingStrainRate_Files\\Results\\NADH_Micron\\" +List_Sheets[i]+'_NADH_Time'+
                str(i)+'.csv') is True:
        # Delete the previous Strains.csv
        os.remove(base + "ProcessingStrainRate_Files\\Results\\NADH_Micron\\" +List_Sheets[i]+'_NADH_Time'+
                str(i)+'.csv')
        df3.to_csv(base + "ProcessingStrainRate_Files\\Results\\NADH_Micron\\" +List_Sheets[i]+'_NADH_Time'+
                str(i)+'.csv', index=False, mode='a',header=True)
    else:
        # Turn into a dataframe to save the file.
        df3.to_csv(base + "ProcessingStrainRate_Files\\Results\\NADH_Micron\\" +List_Sheets[i]+'_NADH_Time'+
                str(i)+'.csv', index=False, mode='a',header=True)

In [ ]:
###############                   APPLY CODE AND GENERATE DATA        #########################

# Nota bene: When I run this code, there seems to be an error with python loading files from folders.
# If I run this block again, the error for that specific directory disappears. So I run this block 13 times

#for i in range(1):
for i in range(len(List_folder)):
    folder = List_folder[i]
    ## Import tif datasets:
    fn,images = compileFolders(i)
    
    # Find shortest kymograph length in lists
    minP = shortL(fn)
    
    # Iterate through array of filenames and create one averaged master array of microscope intensity values
    ToFunc = avgCond(images,minP)
    
    #Create a variable that has NADH concentration from Microscope Intensity
    ToGraph = to_NADH(ToFunc)
    
    # Make an array of distances in microns
    micronList = micronArr(minP)
    
    # Create a figure in the appropriate place to measure the experiment results by eye.
    saveFig(ToGraph[::10],micronList,i)
    
    # Save data as excel spreadsheets, in "ProcessingStrainRate_Files\Results\NADH_Micron\" with the format NADH_Time_0.csv
    saveData(ToGraph,micronList,i)

In [ ]:
# Feedback from Jose:
# We want to create 1D Reaction-diffusion model for NADH consumption 
# Goals: Get diffusion constant and rate of ATPase activity by myosin.
# Average kymographs together and then plot/do data processing 
# Make sure kymograph data contain information on how it was drawn.

# Futher thinking from Francis:
# Fix units in y-axis for kymograph
# Set value of y-axis as NADH concentration

In [ ]:
# 10/2/2023 Updates:
#Successfully coded for all datapoints and included a smoothing function. (Awesome!)
# Now need to update code again to run through all folders and also save excel spreadsheets containing NADH and distance data.

# 10/3/2023 Updates:
# Added Title to NADH graph
# Created subfolder for NADH graph

# 4/10/2023 Updates:
# Inspect and update code for final data run. Write through data processing document describing waht work is possible.